<a href="https://colab.research.google.com/github/appling2024/MSP/blob/Liza_L/SemAn/W2V.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- типы ДСМ
- принцип работы W2V
- cbow и skip-gram

## 1. Предобработка корпуса

Для предобработки мы будем использовать [*UDPipe*](https://ufal.mff.cuni.cz/udpipe)

In [ ]:
!pip install wget

  Preparing metadata (setup.py) ... done
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9655 sha256=917d30730229ca60b6834cdf3ca955dbe1c745c81c4c10e28ec10f8388415657
  Stored in directory: /root/.cache/pip/wheels/40/b3/0f/a40dbd1c6861731779f62cc4babcb234387e11d697df70ee97
Successfully built wget


In [ ]:
import wget
import sys

udpipe_url = 'https://rusvectores.org/static/models/udpipe_syntagrus.model'

modelfile = wget.download(udpipe_url)
print('ok')

ok


Функция для предобработки текста

In [ ]:
def process(pipeline, text='Строка', keep_pos=True, keep_punct=False):
    entities = {'PROPN'}
    named = False
    memory = []
    mem_case = None
    mem_number = None
    tagged_propn = []

    # обрабатываем текст, получаем результат в формате conllu:
    processed = pipeline.process(text)

    # пропускаем строки со служебной информацией:
    content = [l for l in processed.split('\n') if not l.startswith('#')]

    # извлекаем из обработанного текста леммы, тэги и морфологические характеристики
    tagged = [w.split('\t') for w in content if w]

    for t in tagged:
        if len(t) != 10:
            continue
        (word_id, token, lemma, pos, xpos, feats, head, deprel, deps, misc) = t
        if not lemma or not token:
            continue
        if pos in entities:
            if '|' not in feats:
                tagged_propn.append('%s_%s' % (lemma, pos))
                continue
            morph = {el.split('=')[0]: el.split('=')[1] for el in feats.split('|')}
            if 'Case' not in morph or 'Number' not in morph:
                tagged_propn.append('%s_%s' % (lemma, pos))
                continue
            if not named:
                named = True
                mem_case = morph['Case']
                mem_number = morph['Number']
            if morph['Case'] == mem_case and morph['Number'] == mem_number:
                memory.append(lemma)
                if 'SpacesAfter=\\n' in misc or 'SpacesAfter=\s\\n' in misc:
                    named = False
                    past_lemma = '::'.join(memory)
                    memory = []
                    tagged_propn.append(past_lemma + '_PROPN ')
            else:
                named = False
                past_lemma = '::'.join(memory)
                memory = []
                tagged_propn.append(past_lemma + '_PROPN ')
                tagged_propn.append('%s_%s' % (lemma, pos))
        else:
            if not named:
                if pos == 'NUM' and token.isdigit():  # Заменяем числа на xxxxx той же длины
                    continue
                tagged_propn.append('%s_%s' % (lemma, pos))
            else:
                named = False
                past_lemma = '::'.join(memory)
                memory = []
                tagged_propn.append(past_lemma + '_PROPN ')
                tagged_propn.append('%s_%s' % (lemma, pos))

    if not keep_punct:
        tagged_propn = [word for word in tagged_propn if word.split('_')[1] != 'PUNCT']
    if not keep_pos:
        tagged_propn = [word.split('_')[0] for word in tagged_propn]
    return tagged_propn

print('ok')


ok


In [ ]:
!pip install ufal.udpipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 936.8/936.8 kB 32.7 MB/s eta 0:00:00


In [ ]:
from ufal.udpipe import Model, Pipeline
import os
import re

def tag_ud(text='Текст нужно передать функции в виде строки!', modelfile='udpipe_syntagrus.model'):
    cnt = 0
    udpipe_model_url = 'https://rusvectores.org/static/models/udpipe_syntagrus.model'
    udpipe_filename = udpipe_model_url.split('/')[-1]

    if not os.path.isfile(modelfile):
        print('UDPipe model not found. Downloading...', file=sys.stderr)
        wget.download(udpipe_model_url)

    print('\nLoading the model...', file=sys.stderr)
    model = Model.load(modelfile)
    process_pipeline = Pipeline(model, 'tokenize', Pipeline.DEFAULT, Pipeline.DEFAULT, 'conllu')

    print('Processing input...', file=sys.stderr)
    lines = text.split('\n')
    tagged = []
    for line in lines:
        # line = unify_sym(line.strip()) # здесь могла бы быть ваша функция очистки текста
        output = process(process_pipeline, text=line)
        tagged_line = ' '.join(output)
        tagged.append(tagged_line)
        cnt += 1
        if cnt%1000 == 0:
            print(cnt)
    return '\n'.join(tagged)

Приступаем непосредственно к предобработке нашего корпуса.

In [ ]:
text = open(r'combined1.txt', 'r', encoding='utf-8').read()
processed_text = tag_ud(text=text, modelfile=modelfile)
print(processed_text[:350])
with open('combined1.txt', 'w', encoding='utf-8') as out:
    out.write(processed_text)


Loading the model...
Processing input...


1000
2000
3000
4000
5000
6000
7000
8000
d_X \poem_corpus_1\_NOUN poem_corpus_filtered_1.txt_X
﻿_NOUN ночь_PROPN  Посвещаться_VERB Макс::павлов_PROPN  ночь_PROPN холодный_ADJ воздух_NOUN темнота_NOUN и_CCONJ лишь_PART луна_NOUN так_ADV высокий_ADJ и_CCONJ ты_PRON идти_VERB и_PART ты_PRON один_ADJ и_CCONJ думать_VERB что_PRON не_PART нужный_ADJ и_CCONJ словно_PART сон_NOUN стук_NOUN каблук


In [ ]:
!pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 14.0 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.15.3
    Uninstalling scipy-1.15.3:
      Successfully uninstalled scipy-1.15.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
tsfresh 0.21.0 requires scipy>=1.14.0;

In [ ]:
import sys
import gensim, logging

logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

Одна строка - одно предложение

In [ ]:
f = 'combined1.txt'
data = gensim.models.word2vec.LineSentence(f)

2. Приступаем к обучению моделей

2.1 CBOW

* window - размер окна наблюдения, min_count - мин. частотность слова
0 - CBOW, 1 - Skip-gram

In [ ]:
model_CBOW = gensim.models.Word2Vec(data, vector_size=500, window=5, min_count=2, sg=0)

Сохраняем модель

In [ ]:
model_CBOW.save('cbow.model')

Загружаем сохраненную модель

In [ ]:
from gensim.models import Word2Vec
cbow = Word2Vec.load('cbow.model')

Косинусное сходство

In [ ]:
cbow.wv.similarity("ночь_NOUN", "холодный_ADJ")

0.8365321

In [ ]:
cbow.wv.similarity("ночь_NOUN", "темнота_NOUN")

0.8053577

In [ ]:
cbow.wv.similarity("ночь_NOUN", "луна_NOUN")

0.8485318

In [ ]:
for t in cbow.wv.most_similar(positive=[u'ночь_NOUN'], topn=20):
    print (t[0], t[1])

рассвет_NOUN 0.9283241629600525
вечер_NOUN 0.9117879867553711
весна_NOUN 0.899536669254303
сон_NOUN 0.8969354629516602
осень_NOUN 0.8914254307746887
ярко_ADJ 0.8869689106941223
приехать_VERB 0.8867573738098145
лето_NOUN 0.8841565847396851
последний_ADJ 0.8691099286079407
тень_NOUN 0.866465151309967
тишина_NOUN 0.865306556224823
дождь_NOUN 0.8631188273429871
помчусь_VERB 0.853607714176178
утро_NOUN 0.8525328040122986
луна_NOUN 0.8485317826271057
просыпаться_VERB 0.8483776450157166
долететь_PROPN 0.8465686440467834
снова_ADV 0.843372642993927
бессонный_ADJ 0.8418796062469482
птица_NOUN 0.841102659702301


Евклидово расстояние

In [ ]:
import numpy as np

euclid1 = np.linalg.norm(cbow.wv['ночь_NOUN'] - cbow.wv['луна_NOUN'])
euclid2 = np.linalg.norm(cbow.wv['ночь_NOUN'] - cbow.wv['обед_NOUN'])
print(euclid1, euclid2)

4.3724227 6.6024294


## 2.2 Skip-Gram

In [ ]:
model_sg = gensim.models.Word2Vec(data, vector_size=500, window=5, min_count=2, sg=1)

In [ ]:
model_sg.save('skip-gram.model')

In [ ]:
from gensim.models import Word2Vec
sg = Word2Vec.load('skip-gram.model')

In [ ]:
for t in sg.wv.most_similar(positive=[u'ночь_NOUN'], topn=20):
    print (t[0], t[1])

ночь_PROPN 0.7846153378486633
сон_PROPN 0.763526976108551
вечер_NOUN 0.739786684513092
бессонный_ADJ 0.7294158935546875
тишина_NOUN 0.7242397665977478
мрачный_ADJ 0.7208986878395081
призрак_NOUN 0.7200158834457397
ночной_ADJ 0.7175204753875732
вчерашний_ADJ 0.7147084474563599
вылетать_VERB 0.7104833126068115
ночка_NOUN 0.7081984877586365
спаться_VERB 0.7071475386619568
кошмар_NOUN 0.7069290280342102
подушка_NOUN 0.7060327529907227
луна_NOUN 0.7056245803833008
утро_PROPN 0.7050511837005615
уснуть_VERB 0.7014286518096924
неслышно_ADV 0.7005112767219543
рассвет_PROPN 0.70004802942276
мрак_NOUN 0.6989622116088867


In [ ]:
sg.wv.similarity("ночь_NOUN", "холодный_ADJ")

0.6121034

In [ ]:
sg.wv.similarity("ночь_NOUN", "темнота_NOUN")

0.6474276

In [ ]:
sg.wv.similarity("ночь_NOUN", "луна_NOUN")

0.70562464

In [ ]:
euclid3 = np.linalg.norm(sg.wv['ночь_NOUN'] - sg.wv['луна_NOUN'])
euclid4 = np.linalg.norm(sg.wv['ночь_NOUN'] - sg.wv['обед_NOUN'])
print(euclid3, euclid4)

2.4048855 2.804456


## Коллокаты

In [ ]:
import re

for t in sg.wv.most_similar(positive=[u'ночь_NOUN'], topn=10):
  cond = re.search(r'_(NOUN)|(ADJ)|(NUM)', t[0])
  if cond != None:
    print (t[0], t[1])

вечер_NOUN 0.739786684513092
бессонный_ADJ 0.7294158935546875
тишина_NOUN 0.7242397665977478
мрачный_ADJ 0.7208986878395081
призрак_NOUN 0.7200158834457397
ночной_ADJ 0.7175204753875732
вчерашний_ADJ 0.7147084474563599


In [ ]:
import re

for t in cbow.wv.most_similar(positive=[u'ночь_NOUN'], topn=10):
  cond = re.search(r'_(NOUN)|(ADJ)|(NUM)', t[0])
  if cond != None:
    print (t[0], t[1])

рассвет_NOUN 0.9283241629600525
вечер_NOUN 0.9117879867553711
весна_NOUN 0.899536669254303
сон_NOUN 0.8969354629516602
осень_NOUN 0.8914254307746887
ярко_ADJ 0.8869689106941223
лето_NOUN 0.8841565847396851
последний_ADJ 0.8691099286079407
тень_NOUN 0.866465151309967
